# EDA — Stamps ZTF real/bogus

Exploración del dataset descargado con `data/download_alerce.py`.

**Etiquetas:** vienen del `stamp_classifier` de ALeRCE, el único de sus clasificadores con clase `bogus`.
Mapeo binario: `bogus` → 0, y `SN`/`AGN`/`VS`/`asteroid` → 1 (real). Ver `docs/alerce_api.md`.

**Lo que queremos sacar en limpio (decide la Fase 2):**
1. ¿El dataset está balanceado?
2. ¿Qué hacemos con los NaN?
3. ¿Qué normalización necesitan los píxeles?
4. ¿Son todas las stamps 63x63?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.visualization import ZScaleInterval

# El notebook vive en notebooks/, los datos en data/raw/ desde la raiz del repo.
RAW_DIR = Path("..") / "data" / "raw"
CHANNEL_NAMES = ["science", "template", "difference"]
LABEL_NAMES = {0: "bogus", 1: "real"}

meta = pd.read_csv(RAW_DIR / "metadata.csv")
print(f"{len(meta)} objetos en metadata.csv")
meta.head()

In [ ]:
def load_stamp(oid, label):
    """Carga un array (3, 63, 63) desde data/raw/<label>/<oid>.npy."""
    return np.load(RAW_DIR / str(label) / f"{oid}.npy")


# Cargamos todo en memoria: ~1000 stamps x 3 x 63 x 63 x 4 bytes = ~48 MB, entra sin problema.
stamps = {row.oid: load_stamp(row.oid, row.label) for row in meta.itertuples()}
print(f"{len(stamps)} stamps cargadas en memoria")

## 1. Balance de clases

El script pide la misma cantidad de bogus que de real (las 4 clases reales se submuestrean a n/4
cada una), así que el binario debería quedar ~1:1. Igual conviene mirar el desglose por clase
original: si una clase real quedó corta, ALeRCE no tenía suficientes objetos con esa probabilidad.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

binary_counts = meta["label"].map(LABEL_NAMES).value_counts()
binary_counts.plot.bar(ax=axes[0], color=["#c44e52", "#4c72b0"], rot=0)
axes[0].set_title("Balance binario (lo que ve la CNN)")
axes[0].set_ylabel("n objetos")

meta["stamp_class"].value_counts().plot.bar(ax=axes[1], color="#55a868", rot=0)
axes[1].set_title("Clases originales del stamp_classifier")

plt.tight_layout()
plt.show()

ratio = binary_counts.max() / binary_counts.min()
print(binary_counts.to_string())
print(f"\nRatio de desbalance: {ratio:.2f}:1")
print(
    "-> Balanceado, los class weights de la Fase 2 casi no van a corregir nada."
    if ratio < 1.2
    else "-> Hay desbalance: los class weights en la CrossEntropyLoss importan."
)

## 2. Cómo se ven las stamps

Usamos `ZScaleInterval` de astropy, no escala lineal. Las imágenes astronómicas tienen un rango
dinámico enorme (una estrella brillante satura y el resto queda a nivel de fondo), así que con
escala lineal la stamp se ve negra entera. ZScale es el mismo algoritmo que usa DS9 y es lo que
usan los astrónomos para inspeccionar visualmente.

In [ ]:
zscale = ZScaleInterval()


def show_stamp(ax, img, title=""):
    """Dibuja un canal con ZScale. Los NaN salen en rojo para que se vean."""
    finite = img[np.isfinite(img)]
    if finite.size:
        vmin, vmax = zscale.get_limits(finite)
    else:
        vmin, vmax = 0, 1
    cmap = plt.get_cmap("gray").copy()
    cmap.set_bad("red")  # NaN en rojo
    ax.imshow(np.ma.masked_invalid(img), cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
    ax.set_title(title, fontsize=9)
    ax.axis("off")


# Un ejemplo por clase original, los 3 canales.
classes = sorted(meta["stamp_class"].unique())
fig, axes = plt.subplots(len(classes), 3, figsize=(7, 2.4 * len(classes)))
axes = np.atleast_2d(axes)

for i, cls in enumerate(classes):
    row = meta[meta["stamp_class"] == cls].iloc[0]
    arr = stamps[row.oid]
    for j in range(3):
        show_stamp(axes[i, j], arr[j], f"{cls} — {CHANNEL_NAMES[j]}")

plt.suptitle("Una stamp por clase (ZScale; NaN en rojo)", y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
# Varios bogus vs varios real, solo el canal difference: es donde vive la senal del transiente
# (science - template), y donde la diferencia visual entre real y bogus deberia ser mas obvia.
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4.5))

for r, label in enumerate([0, 1]):
    subset = meta[meta["label"] == label].head(n_show)
    for c, row in enumerate(subset.itertuples()):
        show_stamp(axes[r, c], stamps[row.oid][2], f"{row.stamp_class} (p={row.probability:.2f})")
    axes[r, 0].set_ylabel(LABEL_NAMES[label])

plt.suptitle("Canal difference — bogus (arriba) vs real (abajo)", y=1.01)
plt.tight_layout()
plt.show()

## 3. NaNs

Ya sabemos por la verificación de la API que una minoría de stamps trae NaNs, y que cuando los trae
afectan los 3 canales por igual — firma de que la stamp se recortó contra el borde del CCD y se
rellenó con NaN, en vez de píxeles muertos sueltos. Acá lo cuantificamos sobre el dataset completo
para decidir la estrategia de la Fase 2: **descartar** las muy afectadas o **imputar**.

In [ ]:
with_nan = meta[meta["nan_frac"] > 0]
print(f"Stamps con algun NaN: {len(with_nan)}/{len(meta)} ({100 * len(with_nan) / len(meta):.1f}%)")

if len(with_nan):
    print(f"\nEn las afectadas, fraccion de pixeles NaN:")
    print(with_nan["nan_frac"].describe()[["min", "50%", "max"]].to_string())

    # Importa si los NaN atacan mas a una clase que a otra: si fuera asi, la CNN
    # podria aprender "tiene NaN => bogus" en vez de mirar la fisica. Seria un atajo espurio.
    print("\nnan_frac medio por clase binaria:")
    print(meta.groupby(meta["label"].map(LABEL_NAMES))["nan_frac"].agg(["mean", "max"]).to_string())

    fig, ax = plt.subplots(figsize=(7, 3.5))
    for label, color in [(0, "#c44e52"), (1, "#4c72b0")]:
        vals = meta[(meta["label"] == label) & (meta["nan_frac"] > 0)]["nan_frac"]
        if len(vals):
            ax.hist(vals, bins=20, alpha=0.6, label=LABEL_NAMES[label], color=color)
    ax.set_xlabel("fraccion de pixeles NaN")
    ax.set_ylabel("n stamps")
    ax.set_title("Distribucion de NaN (solo las stamps afectadas)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Ninguna stamp tiene NaN en esta descarga.")

## 4. Distribución de valores de píxel → estrategia de normalización

Esto define directamente qué hace el `Dataset` de la Fase 2. Los stamps astronómicos vienen en
cuentas ADU con rango dinámico enorme y outliers fuertes, así que una normalización min-max se
rompe con un solo píxel saturado.

In [ ]:
all_stamps = np.stack(list(stamps.values()))  # (N, 3, 63, 63)
print(f"tensor completo: {all_stamps.shape}, dtype={all_stamps.dtype}")

rows = []
for c, name in enumerate(CHANNEL_NAMES):
    vals = all_stamps[:, c, :, :].ravel()
    finite = vals[np.isfinite(vals)]
    rows.append(
        {
            "canal": name,
            "min": finite.min(),
            "p1": np.percentile(finite, 1),
            "mediana": np.median(finite),
            "p99": np.percentile(finite, 99),
            "max": finite.max(),
            "std": finite.std(),
        }
    )

stats = pd.DataFrame(rows).set_index("canal")
display(stats)

# La razon por la que min-max no sirve: cuanto mas extremo es el max respecto al p99.
print("\nCuantas veces el max supera al p99 (mide cuan pesada es la cola):")
for name, row in stats.iterrows():
    spread = row["max"] / row["p99"] if row["p99"] > 0 else float("inf")
    print(f"  {name:11s}: {spread:8.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for c, (ax, name) in enumerate(zip(axes, CHANNEL_NAMES)):
    vals = all_stamps[:, c, :, :].ravel()
    finite = vals[np.isfinite(vals)]
    # Recortamos a p1-p99 para poder ver algo: con el rango completo, los outliers
    # aplastan el histograma en una sola barra.
    lo, hi = np.percentile(finite, [1, 99])
    ax.hist(finite[(finite >= lo) & (finite <= hi)], bins=80, color="#4c72b0")
    ax.set_title(f"{name} (recortado a p1-p99)", fontsize=10)
    ax.set_yscale("log")

plt.suptitle("Distribucion de valores de pixel por canal", y=1.02)
plt.tight_layout()
plt.show()

## 5. Chequeo de dimensiones

La CNN asume 63x63 fijo (el `Flatten` depende de eso). Si aparece una stamp de otro tamaño,
revienta el `DataLoader` al armar el batch. Mejor enterarse acá.

In [ ]:
shape_counts = meta["shape"].value_counts()
print(shape_counts.to_string())

if len(shape_counts) == 1 and shape_counts.index[0] == "3x63x63":
    print("\nOK: todas las stamps son 3x63x63. El input de la CNN es seguro.")
else:
    print("\nOJO: hay shapes heterogeneos. Hay que resizear/paddear en el Dataset de la Fase 2.")

## Conclusiones → qué implica para la Fase 2

_(completar al correr el notebook sobre el dataset final)_

| Hallazgo | Qué hacer en la Fase 2 |
|---|---|
| Balance binario | Dataset balanceado por construcción → los class weights quedan cerca de 1. Se dejan igual, porque el ratio real de ZTF en producción sí está desbalanceado. |
| NaNs por recorte en el borde del CCD | Registrar `nan_frac`; descartar las stamps con `nan_frac` alto e imputar el resto con la mediana del canal. **Verificar que los NaN no correlacionen con la clase**, o la CNN aprende el atajo "NaN ⇒ bogus". |
| Colas pesadas en los píxeles | Nada de min-max. Usar clipping por percentil (p1–p99) + estandarización por canal, o escala tipo `arcsinh`. |
| Dimensiones | Confirmar 63x63 uniforme. |

**Augmentation** (justificación para el README): flips y rotaciones de 90° son válidos acá porque
el cielo no tiene orientación preferente — la etiqueta real/bogus es invariante a rotar la imagen.
No aplican shears ni cambios de escala: deformarían la PSF, que es justamente una de las señales
que distingue un transiente real de un artefacto.